# Assignment 2A — Part C: Preference Alignment via DPO  [7 Marks]

The Part A and Part B outputs become a preference dataset; the Assignment 1 QLoRA adapter is then
fine-tuned with DPO and compared against itself on five unseen queries.

| Step | Deliverable | Marks |
|---|---|---|
| C1 | ≥60 preference pairs (`prompt`, `chosen`, `rejected`) + counts and length delta | 2 |
| C2 | DPO fine-tuning, `beta=0.1`, 3 epochs, loss + reward margin per epoch, `dpo_adapter/` | 3 |
| C3 | 5-query SFT vs DPO comparison table + 100–150 word analysis | 2 |


> **Backend switch.** Every cell runs with `BACKEND = "offline"`, a deterministic
> corpus-grounded simulator, so the tables, the chain comparison, the PAL execution and the
> preference dataset can be produced and checked on CPU with no model download. For the marked
> run, open a GPU runtime, set `BACKEND = "adapter"`, and re-run; the code path is identical and
> every saved artefact records which backend produced it.


In [ ]:
# Run once in Colab:
# !pip install -q -r assignment2a/requirements_colab.txt


In [ ]:
from pathlib import Path
import sys, json

A2 = Path.cwd()
if (A2 / "assignment2a").exists():
    A2 = A2 / "assignment2a"
sys.path.insert(0, str(A2 / "src"))

import a2_paths
from a2_paths import DEFAULT_SFT_ADAPTER, base_model_source, sft_adapter_path
from domain_lm import DomainLM, wrap

# "offline" = deterministic CPU simulator for local validation.
# "adapter" = real generation from the Assignment 1 QLoRA adapter (needs a GPU runtime).
BACKEND = "offline"
SFT_ADAPTER = DEFAULT_SFT_ADAPTER

ARTIFACTS = A2 / "outputs"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

lm = DomainLM(
    base_model_id=base_model_source(),
    adapter_path=sft_adapter_path(SFT_ADAPTER) if BACKEND == "adapter" else None,
    backend=BACKEND,
    label=f"sft_{SFT_ADAPTER}",
)
print(json.dumps(lm.describe(), indent=2))


## C1 — Preference dataset creation

- `chosen` — the stronger output: the CoT answer, the self-consistency majority vote, the
  instruction-prompted answer, the self-reflection revision, or the APE-selected answer.
- `rejected` — the weaker output: the zero-shot, single-pass answer for the *same* prompt.
- A pair is kept only when the shared rubric prefers `chosen` by at least 3 points, so the labels
  agree with the comparison tables in Parts A and B.
- The five Part C3 evaluation queries are excluded from the prompt pool, so C3 is a genuinely
  held-out test.


In [ ]:
import pandas as pd
import preference_data as pdata
from a2_paths import PREFERENCE_DATASET

REBUILD = False  # set True to regenerate from this notebook's backend
if REBUILD or not PREFERENCE_DATASET.exists():
    pairs, stats = pdata.build_preference_pairs(lm)
    pdata.save_jsonl(pairs, PREFERENCE_DATASET)
else:
    pairs = pdata.load_jsonl(PREFERENCE_DATASET)
    stats = pdata.summarise(pairs)
print(json.dumps(stats, indent=2))


In [ ]:
problems = pdata.validate(pairs, min_pairs=60)
print("validation problems:", problems or "none")
assert not problems, problems
print(f"\n{stats['pairs']} pairs over {stats['unique_prompts']} prompts")
print(f"avg chosen length {stats['avg_chosen_words']} words, "
      f"avg rejected length {stats['avg_rejected_words']} words, "
      f"delta {stats['avg_length_delta_words']:+} words")


In [ ]:
display(pd.DataFrame(
    [{"chosen_source": k, "pairs": v} for k, v in stats["pairs_by_chosen_source"].items()]
))

sample = pairs[0]
print("PROMPT:\n", wrap(sample["prompt"]))
print("\nREJECTED (zero-shot):\n", wrap(sample["rejected"]))
print("\nCHOSEN (" + sample["chosen_source"] + "):\n", wrap(sample["chosen"]))
print("\nscore margin:", sample["score_margin"])


**Length-delta caveat.** The chosen responses are systematically longer than the rejected ones.
DPO can exploit that and learn "longer is better" instead of "better reasoning is better". Two
mitigations are in place: the rubric caps the completeness reward at 220 words so padding is not
rewarded, and the C3 improvement test in `dpo_training._improvement_reason` explicitly flags a row
as *not improved* when the DPO answer is much longer without a gain in grounding or structure.


## C2 — DPO fine-tuning

The Assignment 1 adapter is loaded as the SFT policy; TRL uses the frozen base + adapter as the
implicit reference model. `beta=0.1`, 3 epochs, as specified.


In [ ]:
import dpo_training as dpo
from a2_paths import DPO_ADAPTER_DIR

print(json.dumps(dpo.DPO_CONFIG, indent=2))
print("SFT adapter:", sft_adapter_path(SFT_ADAPTER), "| exists:", sft_adapter_path(SFT_ADAPTER).exists())
print("DPO output :", DPO_ADAPTER_DIR)

RUN_DPO = False  # set True on a GPU runtime after Assignment 1 Part B has produced the adapter


In [ ]:
if RUN_DPO:
    result = dpo.train_dpo(PREFERENCE_DATASET, sft_adapter=SFT_ADAPTER, output_dir=DPO_ADAPTER_DIR)
    log_history = json.loads((DPO_ADAPTER_DIR / "dpo_log_history.json").read_text(encoding="utf-8"))
    print(json.dumps(result["config"], indent=2))
else:
    log_path = DPO_ADAPTER_DIR / "dpo_log_history.json"
    log_history = json.loads(log_path.read_text(encoding="utf-8")) if log_path.exists() else []
    print("DPO training not run. Set RUN_DPO=True on a GPU runtime.")


In [ ]:
if log_history:
    epochs = dpo.epoch_metrics(log_history)
    display(pd.DataFrame(epochs))
    print("DPO loss should decrease across epochs and the reward margin should increase.")
else:
    display(pd.DataFrame([
        {"epoch": 1, "dpo_loss": None, "reward_chosen": None, "reward_rejected": None,
         "reward_margin": None, "reward_accuracy": None},
        {"epoch": 2, "dpo_loss": None, "reward_chosen": None, "reward_rejected": None,
         "reward_margin": None, "reward_accuracy": None},
        {"epoch": 3, "dpo_loss": None, "reward_chosen": None, "reward_rejected": None,
         "reward_margin": None, "reward_accuracy": None},
    ]))
    print("Fill this table from the GPU run.")


In [ ]:
import matplotlib.pyplot as plt

if log_history:
    epochs = dpo.epoch_metrics(log_history)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot([e["epoch"] for e in epochs], [e["dpo_loss"] for e in epochs], marker="o")
    axes[0].set_title("DPO loss per epoch"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
    axes[1].plot([e["epoch"] for e in epochs], [e["reward_chosen"] for e in epochs],
                 marker="o", label="chosen")
    axes[1].plot([e["epoch"] for e in epochs], [e["reward_rejected"] for e in epochs],
                 marker="o", label="rejected")
    axes[1].plot([e["epoch"] for e in epochs], [e["reward_margin"] for e in epochs],
                 marker="s", linestyle="--", label="margin")
    axes[1].set_title("Implicit rewards per epoch"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout(); plt.show()
else:
    print("No DPO log history yet; run C2 on a GPU runtime to produce the curves.")


## C3 — Alignment evaluation on five unseen queries

Both adapters answer the same bare zero-shot prompt. That is the point of the test: alignment
should make the model produce the *preferred* shape of answer without any prompt scaffolding.


In [ ]:
from assignment2_queries import HELD_OUT_QUERIES

for i, q in enumerate(HELD_OUT_QUERIES, 1):
    print(f"{i}. {q}")
assert not set(q.lower() for q in HELD_OUT_QUERIES) & set(p["prompt"].lower() for p in pairs)
print("\nconfirmed: none of these queries appears in the preference dataset")


In [ ]:
sft_lm = lm
if BACKEND == "adapter":
    dpo_lm = DomainLM(base_model_id=base_model_source(), adapter_path=DPO_ADAPTER_DIR,
                      backend="adapter", label="dpo_adapter")
else:
    # Offline stand-in for the aligned adapter: it answers a bare prompt in the preferred shape.
    dpo_lm = DomainLM(backend="offline", label="dpo_adapter", aligned=True)

rows_c3 = dpo.compare_adapters(sft_lm, dpo_lm, HELD_OUT_QUERIES)
summary = dpo.comparison_summary(rows_c3)
print(json.dumps(summary, indent=2))


In [ ]:
for r in rows_c3:
    print("=" * 100)
    print(f"Query {r['n']}: {r['query']}")
    print(f"\n[SFT {r['sft_score']['total']}]\n", wrap(r["sft_output"]))
    print(f"\n[DPO {r['dpo_score']['total']}]\n", wrap(r["dpo_output"]))
    print(f"\nImprovement: {r['improved']} — {r['reason']}")


In [ ]:
display(pd.DataFrame([{
    "Query": f"Query {r['n']}",
    "SFT score": r["sft_score"]["total"],
    "DPO score": r["dpo_score"]["total"],
    "Delta": round(r["dpo_score"]["total"] - r["sft_score"]["total"], 2),
    "Improvement?": r["improved"],
    "Reason": r["reason"],
} for r in rows_c3]))

table_md = dpo.markdown_table(rows_c3)
(ARTIFACTS / "part_c_comparison.md").write_text(table_md, encoding="utf-8")
print(table_md)


In [ ]:
artefact = {
    "backend_sft": sft_lm.describe(),
    "backend_dpo": dpo_lm.describe(),
    "c1_stats": stats,
    "c2_config": dpo.DPO_CONFIG,
    "c2_epoch_metrics": dpo.epoch_metrics(log_history) if log_history else [],
    "c3_rows": rows_c3,
    "c3_summary": summary,
}
path = ARTIFACTS / "part_c_results.json"
path.write_text(json.dumps(artefact, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("saved", path)


## C3 — Alignment analysis (100–150 words)

> **Rewrite this paragraph from your own GPU run before submitting.** Cite the actual row numbers
> and scores printed above; the sentences below show the required shape of the argument.

DPO alignment improved domain response quality on the held-out set: mean rubric score rose from
the SFT adapter to the DPO adapter, and the gain came almost entirely from *shape* rather than from
new knowledge. The largest improvements were on Query 1 and Query 3, where the SFT adapter returned
a single definitional sentence and the DPO adapter returned the definition plus the governing
considerations, the limiting condition, and the educational caveat — exactly the pattern carried by
the `chosen` responses. The weakest case was the query whose answer the SFT adapter already covered
well; there the DPO output was longer without adding corpus content, which is the expected
length-bias failure mode of preference training. DPO therefore transferred the *format and caveat
discipline* of CoT and self-reflection into the weights, but it did not add factual coverage that
was absent from the Assignment 1 corpus.

### Caveats to state in the report

1. The preference pairs were generated and labelled by the same rubric that scores C3, so the
   evaluation is favourable by construction. An independent human rating of a 20-row sample is the
   honest check.
2. `chosen` is systematically longer than `rejected`, so some of the gain is length bias.
3. 95 pairs over 19 prompts is a small, narrow dataset; the gains should not be extrapolated beyond
   this corpus.
